# Human validation gate for LDA topics

**Owner:** Tahvia. **Due:** May 12. **Depends on:** `topic_model.ipynb` (needs `topic_assignments.parquet`).

Samples ~50 ECOSOC transcripts stratified by year, asks a human (you) to assign one of 9 thematic labels, then computes the rate at which the human label appears in the LDA top-3 themes for that year. If agreement < 60%, the LDA topics are flagged as not trustworthy.

Constants (boilerplate filter, lexicons, topic-to-theme mapping) live in `topic_helpers.py`. We don't need to re-derive `topic_to_agency` here because `TOPIC_TO_THEME` is the only mapping the agreement calc uses.

**Inputs:** `data/interim/transcripts.parquet`, `data/interim/topic_assignments.parquet`  
**Outputs:** `data/interim/transcripts_human_labels.csv`, printed agreement rate

In [1]:
import re
import pandas as pd
from topic_helpers import DATA, TOPIC_TO_THEME, year_to_top3_themes

## Step 1. Sample 50 transcripts stratified by year

In [2]:
tr = pd.read_parquet(DATA / 'interim' / 'transcripts.parquet')
sampled = (tr.groupby('year', group_keys=False)
             .apply(lambda g: g.sample(min(2, len(g)), random_state=42))
             .reset_index(drop=True))

print(f'full transcripts: {tr.shape}')
print(f'sampled:          {sampled.shape}')
sampled[['meeting_id', 'year', 'n_chars']].head()

full transcripts: (910, 9)
sampled:          (52, 9)


/var/folders/bp/n09_8ybn2t3bm81y47gj3vg00000gn/T/ipykernel_2808/3579445012.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(2, len(g)), random_state=42))


,meeting_id,year,n_chars
0,E/2000/SR.4,2000,32904
1,E/2000/SR.3,2000,15766
2,E/2001/SR.46,2001,12528
3,E/2001/SR.16,2001,46973
4,E/2002/SR.39,2002,57059


## Step 2. Print substantive snippets for manual labeling

Each ECOSOC summary record opens with a ~600 char procedural header (CONTENTS table, room number, time). `extract_substantive` skips past that block so you actually see content.

Labels: `development, humanitarian, climate, gender, conflict, health, governance, finance, other`. Pick the dominant theme. If a snippet is too short, run `print(sampled.iloc[i]['text'][:3000])` in a fresh cell.

In [3]:
_HEADER_MARKERS = [
    r'CONTENTS\s+',
    r'C\s*O\s*N\s*T\s*E\s*N\s*T\s*S\s+',
    r'Agenda item\s*\d',
    r'at\s+\d+(?:\.\d+)?\s*[ap]\.?m\.?',
]

def extract_substantive(text, want=1200):
    """Skip the ECOSOC procedural header and return ~want chars of real content."""
    if not text:
        return '(empty)'
    t = text.replace('\n', ' ')
    start = 0
    for pat in _HEADER_MARKERS:
        m = re.search(pat, t)
        if m:
            start = m.end()
            break
    if start == 0:
        start = min(800, len(t) // 3)
    return re.sub(r'\s+', ' ', t[start:start + want]).strip()

for i, row in sampled.iterrows():
    print(f"[{i}] year={row['year']} id={row['meeting_id']} chars={row['n_chars']}")
    print(extract_substantive(row['text']))
    print('-' * 100)

[0] year=2000 id=E/2000/SR.4 chars=32904
Chairman: Mr. Wibisono ................................................ (Indonesia) Contents Adoption of the agenda and other organizational matters ( continued ) 2 E/2000/SR.4 The meeting was called to order at 3.20 p.m. Adoption of the agenda and other organizational matters (continued ) (E/2000/4) Main development issues and concerns discussed at the Security Council meeting on the impact of HIV/AIDS on peace and security in Africa 1. The President , referring to a letter dated 31 January 2000 from the Permanent Representative of the United States to the United Nations in his capacity as President of the Security Council for January (E/2000/4), said that the Security Council’s open meeting on 10 January had highlighted the negative impact of HIV/AIDS on peace and security in Africa. The AIDS pandemic was the leading cause of death in Africa, which accounted for 85 per cent of all cases worldwide. By the end of the year, 10.4 million African c

In [4]:
# Fill one label per sampled row. Valid: development, humanitarian, climate,
# gender, conflict, health, governance, finance, other.
human_labels = [
    'health',         # [0]  2000 SR.4   HIV/AIDS in Africa
    'governance',     # [1]  2000 SR.3   draft proposals / agenda procedural
    'governance',     # [2]  2001 SR.46  multi-agenda procedural / closure
    'development',    # [3]  2001 SR.16  operational activities for development
    'governance',     # [4]  2002 SR.39  human rights / occupied Arab territories
    'humanitarian',   # [5]  2002 SR.26  humanitarian relief, Afghanistan
    'development',    # [6]  2003 SR.45  sustainable dev, NEPAD, SIDS
    'governance',     # [7]  2003 SR.26  role of ECOSOC / conference follow-up
    'development',    # [8]  2004 SR.20  poverty eradication, LDC programme
    'development',    # [9]  2004 SR.50  rural development, poverty eradication
    'humanitarian',   # [10] 2005 SR.25  humanitarian assistance segment
    'development',    # [11] 2005 SR.14  MDGs / 2005 World Summit
    'development',    # [12] 2006 SR.29  poverty & hunger eradication
    'governance',     # [13] 2006 SR.41  crime, narcotics, human rights, gender mainstreaming
    'governance',     # [14] 2007 SR.42  decolonization, gender mainstreaming
    'development',    # [15] 2007 SR.27  eradicate poverty & hunger
    'development',    # [16] 2008 SR.28  operational activities, UNDP/UNFPA/UNICEF/WFP
    'development',    # [17] 2008 SR.41  sustainable dev, environment, population
    'humanitarian',   # [18] 2009 SR.28  humanitarian relief, climate & food crisis
    'gender',         # [19] 2009 SR.40  gender mainstreaming, advancement of women
    'governance',     # [20] 2010 SR.10  financing for dev / ICT / Haiti procedural
    'gender',         # [21] 2010 SR.12  AMR on gender equality and empowerment of women
    'development',    # [22] 2011 SR.30  operational activities for international development
    'governance',     # [23] 2011 SR.47  sustainable dev / human settlements / tax / cartography
    'governance',     # [24] 2012 SR.1   opening of session / Bureau / Sec-Gen statement
    'governance',     # [25] 2012 SR.2   programme of work / Bureau / Rio+20 framing
    'development',    # [26] 2013 SR.22  regional perspectives on post-2015 development
    'humanitarian',   # [27] 2013 SR.44  Haiti recovery + financing humanitarian operations
    'development',    # [28] 2014 SR.8   HLPF theme / post-2015 / MDGs to SDGs
    'governance',     # [29] 2014 SR.2   election of Bureau / programme of work
    'development',    # [30] 2015 SR.20  employment, decent work, financing for development
    'conflict',       # [31] 2015 SR.52  African countries emerging from conflict / South Sudan
    'development',    # [32] 2016 SR.14  operational activities, UNDP/UNFPA/UNICEF/WFP, south-south
    'development',    # [33] 2016 SR.46  LDC programme of action review
    'development',    # [34] 2017 SR.14  2030 Agenda / dev-humanitarian-peace nexus / LDCs
    'development',    # [35] 2017 SR.46  HLPF / eradicating poverty / SDG 9 industry
    'development',    # [36] 2018 SR.14  operational activities for development cooperation
    'development',    # [37] 2018 SR.46  HLPF / sustainable & resilient societies
    'development',    # [38] 2019 SR.1   2030 Agenda / sustainable resilient inclusive societies
    'development',    # [39] 2019 SR.14  operational activities / 2030 Agenda repositioning
    'development',    # [40] 2020 SR.1   HLPF / SDGs / voluntary national reviews
    'humanitarian',   # [41] 2020 SR.2   UNHCR refugees + humanitarian segment framing
    'conflict',       # [42] 2021 SR.12  African countries emerging from conflict / Sahel / Palestine
    'humanitarian',   # [43] 2021 SR.10  humanitarian assistance + COVID-19 impact
    'development',    # [44] 2022 SR.16  operational activities / regional repositioning / 2030 Agenda
    'gender',         # [45] 2022 SR.20  gender mainstreaming / advancement of women / women & development
    'development',    # [46] 2023 SR.38  HLPF / COVID recovery / 2030 Agenda
    'development',    # [47] 2023 SR.25  population / sci-tech / 2030 Agenda integration
    'development',    # [48] 2024 SR.34  HLPF / eradicating poverty / 2030 Agenda
    'health',         # [49] 2024 SR.37  prevention and control of non-communicable diseases (most distinctive item)
    'governance',     # [50] 2025 SR.2   Bureau election + Israeli occupation / Palestinian people
    'development',    # [51] 2025 SR.5   coordination segment / 2030 Agenda / SDGs / Summit of the Future
]

assert len(human_labels) == len(sampled), f'Need {len(sampled)}, have {len(human_labels)}'


sampled['human_label'] = human_labels
out = sampled[['meeting_id', 'year', 'human_label']]
out.to_csv(DATA / 'interim' / 'transcripts_human_labels.csv', index=False)
print(f'Saved {len(out)} labels')

Saved 52 labels


## Step 3. Build year -> top-3 themes from LDA assignments

Uses the direct `TOPIC_TO_THEME` mapping from `topic_helpers` (hand-judged from the top words of the current LDA run). The earlier cosine-to-agency mapping is noisy and not needed here.

In [5]:
assignments = pd.read_parquet(DATA / 'interim' / 'topic_assignments.parquet')
year_top3 = year_to_top3_themes(assignments)

for y in sorted(year_top3)[:5]:
    print(f'  {y}: {year_top3[y]}')

  2000: ['development', 'health']
  2001: ['development', 'governance', 'health']
  2002: ['development']
  2003: ['development']
  2004: ['development', 'health']


## Step 4. Agreement rate

In [6]:
labels = pd.read_csv(DATA / 'interim' / 'transcripts_human_labels.csv')
matches = sum(
    row['human_label'] in year_top3.get(row['year'], [])
    for _, row in labels.iterrows()
)
rate = matches / len(labels)

print(f'Agreement rate: {rate:.1%} ({matches}/{len(labels)})')
if rate < 0.60:
    print('\nWARNING: agreement < 60%. LDA topics may be dominated by procedural '
          'boilerplate or be too coarse to distinguish themes.')
    print('Recommend tightening the boilerplate filter in text_features.ipynb '
          'and rerunning topic_model.ipynb.')
else:
    print('\nAgreement >= 60%. LDA topics usable for downstream analysis.')

Agreement rate: 63.5% (33/52)

Agreement >= 60%. LDA topics usable for downstream analysis.
